# 05 · A100 profiling, smoke test and precision benchmark (I-3) — no full training

Order (each step writes a JSON report to `MyDrive/VeriHGN/reports/`):
1. **Gate** — refuses to run without a passing notebook-04 report.
2. **Data loading + graph memory** — Drive read, graph build, collate, host→device per sample (≈ 2 min).
3. **Peak VRAM per design** — one full training step (batch = 1 graph, official) on the largest sample of
   each design in FP32 / FP16 / BF16, with GPU utilisation sampled (≈ 2–5 min).
4. **Smoke test** — read-only inspection of the actual run directory, the checkpoint/resume regression tests
   (seconds, includes a CUDA test), then 2 epochs on the full training split and a 3rd epoch in a **new**
   Trainer that must resume from `<run_dir>/checkpoints/last.pt` on Drive (≈ 5–15 min). If an earlier
   session already left a valid epoch-2 checkpoint, it is reused and only the resume is run.
5. **Precision benchmark** — FP32 vs official FP16-AMP vs BF16-AMP, 20 epochs × 2 seeds, plus a repeatability
   check; validation in FP32, test split never touched (≈ 2–4 h; resumable). **Off by default**: set
   `RUN_PRECISION_BENCHMARK = True` only after the smoke test passed and the benchmark is approved.
6. **Persistence check and stage report** — `reports/STAGE_REPORT.md` collects everything.

Batch size stays 1 graph throughout. Metrics here are diagnostics, not results. Full training is **not**
part of this notebook.

In [ ]:
# ── Bootstrap (identical in every notebook; safe to re-run) ──────────────────────
# Mounts YOUR Google Drive (you authorise in the popup; no credentials are stored),
# unpacks the project code from Drive and puts it on the Python path.
import os, sys, zipfile, subprocess
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/VeriHGN'            # persistent project storage
CODE_ZIP   = f'{DRIVE_ROOT}/code/verihgn_code.zip'        # uploaded from the laptop (tools/make_colab_bundle.py)
CODE_DIR   = '/content/verihgn-project'

if not os.path.exists(CODE_ZIP):
    raise FileNotFoundError(f'Upload verihgn_code.zip to {CODE_ZIP} first (see README, "Colab").')
stamp = f'{CODE_DIR}/.zip_mtime'
if not os.path.exists(stamp) or open(stamp).read() != str(os.path.getmtime(CODE_ZIP)):
    subprocess.run(['rm', '-rf', CODE_DIR], check=True)
    with zipfile.ZipFile(CODE_ZIP) as z:
        z.extractall(CODE_DIR)
    open(stamp, 'w').write(str(os.path.getmtime(CODE_ZIP)))
sys.path.insert(0, f'{CODE_DIR}/src')

marker = '/content/.verihgn_deps_ok'
if not os.path.exists(marker):
    from verihgn_repro.colab import install_dependencies
    install_dependencies()
    open(marker, 'w').write('ok')

from verihgn_repro.colab import DriveLayout
L = DriveLayout().ensure()
print('code:', CODE_DIR, '| drive root:', L.root)

In [ ]:
import json, time, torch, psutil
from pathlib import Path
from verihgn_repro.config import load_config
from verihgn_repro.data.dataset import build_experiment_graphs
from verihgn_repro.data.splits import load_manifest
from verihgn_repro.training.trainer import Trainer
from verihgn_repro.training.checkpointing import RunStatus, inspect_run_dir
from verihgn_repro.training.benchmark import measure_peak_memory, run_precision_benchmark
from verihgn_repro.utils import hardware
from verihgn_repro import reporting
from verihgn_repro.colab import write_json
CFG_DIR = Path(CODE_DIR) / 'configs'
SCOPE = 'official9'
summary = json.load(open(L.reports / '04_validation_summary.json'))
assert all(s['passed'] for s in summary), 'GATE: validation has not passed (run notebook 04)'
PHASH = json.load(open(L.reports / f'03_preprocessing_{SCOPE}.json'))['prep_hash']
PROCESSED = L.processed_for(PHASH)
hw = hardware.detect(); prof = hardware.choose_profile(hw)
print(hw); print(prof)
assert hw.device == 'cuda', 'select an A100 runtime'

### Primary configuration and graphs (built in RAM from the processed data on Drive)

In [ ]:
cfg = load_config(CFG_DIR / 'base.yaml')          # official hyper-parameters; primary variant (D3/D4/D5 defaults)
man = load_manifest(L.splits / f'{SCOPE}__{cfg.experiment.split}.json')
t0 = time.time()
data = build_experiment_graphs(PROCESSED, man, grid_features=cfg.experiment.grid_features,
                               label_norm=cfg.experiment.label_norm, cache_dir=None, prep_hash=PHASH)
print(cfg.run_id(), '| c_max:', data['c_max'], '| samples:', {k: len(v) for k, v in data['graphs'].items()},
      f'| built in {time.time()-t0:.0f}s | RAM available {psutil.virtual_memory().available/1e9:.1f} GB')

### 2 · Data loading and graph memory

In [ ]:
dl = reporting.data_loading_profile(PROCESSED, sum(man['designs'].values(), []), device='cuda', per_design=2)
write_json(dl, L.reports / '05_data_loading.json')
import pandas as pd; display(pd.DataFrame([{k: v for k, v in r.items() if k not in ('nodes_mb', 'edges_mb')} for r in dl]))

### 3 · Peak VRAM per design (largest sample of each design, one full training step)

In [ ]:
mon = hardware.GPUMonitor(interval_s=0.5).start()
rows = measure_peak_memory(cfg, data['graphs'], precisions=tuple(prof.candidate_precisions))
util = mon.stop_and_summary()
pm = {'gpu': hw.name, 'total_vram_gb': hw.total_vram_gb, 'batch_size_graphs': 1, 'rows': rows, 'gpu_util': util,
      'all_steps_finite': all(r['loss_finite'] for r in rows),
      'max_peak_gb': max(r['peak_allocated_gb'] for r in rows)}
write_json(pm, L.reports / '05_peak_memory.json')
display(pd.DataFrame(rows)); print(util, '| max peak', pm['max_peak_gb'], 'GB')

### 4a · Smoke run directory — read-only inspection (the actual files on Drive)

In [ ]:
scfg = load_config(CFG_DIR / 'base.yaml', CFG_DIR / 'smoke.yaml')
scfg.experiment.name += '_smoke'
run_dir = L.runs / scfg.run_id()          # checkpoints live in run_dir / 'checkpoints' / 'last.pt'
pre = inspect_run_dir(run_dir)
print('run directory:', run_dir, '| exists:', pre['exists'])
print('files:', json.dumps(pre['files'], indent=1))
print('status:', json.dumps(pre.get('status'), indent=1, default=str))
print('history epochs:', pre.get('history_epochs'))
for name, c in pre['checkpoints'].items():
    print(name, json.dumps(c, indent=1, default=str))

### 4b · Checkpoint/resume regression tests (≈ 1 min; the CUDA test runs only here)

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, '-m', 'pytest', '-q', '-rs', 'tests/test_checkpoint_resume.py'],
                   cwd=CODE_DIR, capture_output=True, text=True)
print(r.stdout[-4000:], r.stderr[-2000:])
assert r.returncode == 0, 'checkpoint/resume tests failed: do not continue'

### 4c · Smoke test: 2 epochs, then a new Trainer must resume epoch 3 from Drive

In [ ]:
ck = pre['checkpoints'].get('last.pt')
if not pre['files']:
    session1 = 'fresh'                    # nothing there yet: run epochs 1-2 now
elif ck and ck['ok'] and ck['epoch'] == 2 and pre.get('history_epochs') == [1, 2]:
    session1 = 'previous_session'         # epochs 1-2 from an earlier Colab session: reuse them, test the resume
else:                                     # anything else is left untouched; a new directory is used
    run_dir = run_dir.with_name(f'{run_dir.name}__rerun_{time.strftime("%Y%m%d_%H%M%S")}')
    session1 = 'fresh'
print('session 1:', session1, '| run directory:', run_dir)
t0 = time.time()
env_session1 = None
if session1 == 'fresh':
    st1 = Trainer(scfg, data['graphs'], run_dir).run(mode='fresh')         # epochs 1-2 (session limit)
else:
    st1 = RunStatus(run_dir).read()
    env_session1 = json.load(open(run_dir / 'env.json')) if (run_dir / 'env.json').exists() else None
mid = inspect_run_dir(run_dir)
assert mid['checkpoints'].get('last.pt', {}).get('ok') and mid['checkpoints']['last.pt']['epoch'] == 2, mid
scfg.training.max_epochs_this_session = None
tr = Trainer(scfg, data['graphs'], run_dir)                                # new object: must resume
st2 = tr.run(mode='resume')                                                # raises if it cannot resume
post = inspect_run_dir(run_dir)
hist = pd.read_csv(run_dir / 'history.csv')
smoke = {'run_dir': str(run_dir), 'session1_source': session1, 'status_after_session1': st1,
         'status_after_resume': st2, 'resume_info': tr.resume_info,
         'checkpoint_after_session1': mid['checkpoints']['last.pt'],
         'checkpoint_after_resume': post['checkpoints'].get('last.pt'),
         'resumed_correctly': (tr.resume_info['resumed'] and tr.resume_info['start_epoch'] == 2
                               and tr.resume_info['checkpoint'] == str(run_dir / 'checkpoints' / 'last.pt')
                               and st2['epoch'] == 3 and hist['epoch'].tolist() == [1, 2, 3]
                               and post['checkpoints']['last.pt']['epoch'] == 3),
         'seconds': round(time.time() - t0), 'history': hist.to_dict('records'),
         'checkpoints': sorted(post['checkpoints']), 'files_before': pre['files'],
         'env_session1': env_session1, 'env': json.load(open(run_dir / 'env.json'))}
write_json(smoke, L.reports / '05_smoke.json')
cols = ['epoch', 'train_loss', 'val_loss', 'val_avg_spearman', 'train_seconds', 'train_peak_allocated_gb',
        'gpu_util_mean', 'grad_norm_mean', 'skipped_nonfinite']
display(hist[[c for c in cols if c in hist.columns]])
print('resume info:', tr.resume_info)
print('resume ok:', smoke['resumed_correctly'])
assert smoke['resumed_correctly']

### 5 · Precision benchmark — measurement only (≈ 2–4 h, resumable)

In [ ]:
RUN_PRECISION_BENCHMARK = False    # set True only after the smoke test passed and the benchmark is approved
if RUN_PRECISION_BENCHMARK:
    bench = run_precision_benchmark(cfg, data['graphs'], L.runs / 'precision_benchmark', epochs=20, seeds=(42, 43),
                                    precisions=tuple(prof.candidate_precisions), repeat_epochs=3)
    write_json(bench, L.reports / '05_precision_benchmark.json')
    display(pd.DataFrame(bench['summary']).T)
    display(pd.DataFrame(bench['repeatability']))

### 6 · Drive persistence check and stage report

In [ ]:
pc = reporting.persistence_check({'processed': PROCESSED, 'splits': L.splits, 'runs': L.runs, 'reports': L.reports})
write_json(pc, L.reports / '05_persistence.json')
print(json.dumps(pc, indent=1))
text = reporting.summarize_reports(L.reports, L.reports / 'STAGE_REPORT.md')
print(text[:4000])
print('\n→ Download MyDrive/VeriHGN/reports/STAGE_REPORT.md (and the JSON files) and share them back.')

### 7 · Flush Drive before leaving (last cell)

Writes to `/content/drive` are uploaded asynchronously. `flush_and_unmount` blocks until everything written in this session has reached Drive. Run it before closing or disconnecting the runtime; re-run the bootstrap cell afterwards if you continue working.

In [ ]:
from google.colab import drive
drive.flush_and_unmount()
print('Drive flushed and unmounted; all checkpoints and reports written in this session are on Drive.')